In [0]:
%sql
USE financial_transactions.bronze

### 1. Formatação e tratamento: users
Converte cada coluna para o tipo correto, adiciona a cidade e o estado mais próximos da localização do cliente e remove o endereço e as coordenadas, que são dados pessoais sensíveis.

In [0]:
%sql
CREATE OR REPLACE TABLE financial_transactions.silver.users AS
WITH clientes AS (
  SELECT
    CAST(id AS INT)               AS id,
    CAST(current_age AS INT)      AS current_age,
    CAST(retirement_age AS INT)   AS retirement_age,
    CAST(birth_year AS INT)       AS birth_year,
    CAST(birth_month AS INT)      AS birth_month,
    gender,
    CAST(latitude AS DOUBLE)      AS latitude,
    CAST(longitude AS DOUBLE)     AS longitude,
    CAST(REPLACE(per_capita_income, '$', '') AS DECIMAL(12,2)) AS per_capita_income,
    CAST(REPLACE(yearly_income, '$', '')     AS DECIMAL(12,2)) AS yearly_income,
    CAST(REPLACE(total_debt, '$', '')        AS DECIMAL(12,2)) AS total_debt,
    CAST(credit_score AS INT)     AS credit_score,
    CAST(num_credit_cards AS INT) AS num_credit_cards,
    _data_ingestao
  FROM financial_transactions.bronze.users
),
cidade_mais_proxima AS (
  SELECT
    c.id,
    REGEXP_REPLACE(p.NAME, ' (city|town|village|CDP|borough|municipality)$', '') AS city,
    p.USPS AS state,
    6371 * 2 * ASIN(SQRT(
        POWER(SIN(RADIANS(CAST(p.INTPTLAT AS DOUBLE) - c.latitude) / 2), 2)
      + COS(RADIANS(c.latitude)) * COS(RADIANS(CAST(p.INTPTLAT AS DOUBLE)))
      * POWER(SIN(RADIANS(CAST(p.INTPTLONG AS DOUBLE) - c.longitude) / 2), 2)
    )) AS distance_km
  FROM clientes c
  CROSS JOIN financial_transactions.bronze.place_2020 p
  QUALIFY ROW_NUMBER() OVER (PARTITION BY c.id ORDER BY distance_km) = 1
)
SELECT
  c.id,
  c.current_age,
  c.retirement_age,
  c.birth_year,
  c.birth_month,
  c.gender,
  l.city,
  l.state,
  ROUND(l.distance_km, 1) AS city_distance_km,
  c.per_capita_income,
  c.yearly_income,
  c.total_debt,
  c.credit_score,
  c.num_credit_cards,
  c._data_ingestao,
  CURRENT_TIMESTAMP() AS _data_processamento
FROM clientes c
LEFT JOIN cidade_mais_proxima l ON c.id = l.id

num_affected_rows,num_inserted_rows


**Ideal:** Quantidade de linhas iguais nas duas e quase todos os clientes a até 10 km da cidade encontrada.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM financial_transactions.bronze.users) AS linhas_bronze,
  COUNT(*)                         AS linhas_silver,
  COUNT_IF(city IS NULL)           AS sem_cidade,
  COUNT_IF(city_distance_km <= 10) AS ate_10km,
  COUNT_IF(city_distance_km > 10)  AS acima_10km,
  MAX(city_distance_km)            AS maior_distancia
FROM financial_transactions.silver.users

linhas_bronze,linhas_silver,sem_cidade,ate_10km,acima_10km,maior_distancia
2000,2000,0,1964,36,25.8


### 2. Formatação e tratamento: cards
Converte cada coluna para o tipo correto, mascara o número do cartão e remove o CVV, que são dados sensíveis.

In [0]:
%sql
CREATE OR REPLACE TABLE financial_transactions.silver.cards AS
SELECT
  CAST(id AS INT)                                        AS id,
  CAST(client_id AS INT)                                 AS client_id,
  card_brand,
  card_type,
  CONCAT('**** **** **** ', RIGHT(card_number, 4))       AS card_number_masked,
  LAST_DAY(TO_DATE(expires, 'MM/yyyy'))                  AS expires_date,
  has_chip = 'YES'                                       AS has_chip,
  CAST(num_cards_issued AS INT)                          AS num_cards_issued,
  CAST(REPLACE(credit_limit, '$', '') AS DECIMAL(12,2))  AS credit_limit,
  TO_DATE(acct_open_date, 'MM/yyyy')                     AS acct_open_date,
  CAST(year_pin_last_changed AS INT)                     AS year_pin_last_changed,
  card_on_dark_web = 'Yes'                               AS card_on_dark_web,
  _data_ingestao,
  CURRENT_TIMESTAMP()                                    AS _data_processamento
FROM financial_transactions.bronze.cards

num_affected_rows,num_inserted_rows


**Ideal:** Quantidade de linhas iguais e 0 em validade, abertura, limite, chip e dark web vazios.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM financial_transactions.bronze.cards) AS linhas_bronze,
  COUNT(*)                          AS linhas_silver,
  COUNT_IF(expires_date IS NULL)    AS validade_vazia,
  COUNT_IF(acct_open_date IS NULL)  AS abertura_vazia,
  COUNT_IF(credit_limit IS NULL)    AS limite_vazio,
  COUNT_IF(has_chip IS NULL)        AS chip_vazio,
  COUNT_IF(card_on_dark_web IS NULL) AS dark_web_vazio
FROM financial_transactions.silver.cards

linhas_bronze,linhas_silver,validade_vazia,abertura_vazia,limite_vazio,chip_vazio,dark_web_vazio
6146,6146,0,0,0,0,0


### 3. Formatação e tratamento: transactions
Converte cada coluna para o tipo correto, separa o país do estado, identifica compras online e no exterior, preenche os erros vazios com "Nenhum" e marca as compras de chip com cidade "ONLINE".

> **Ajuste após a validação:** a completude mostrou estado e CEP vazios. A investigação confirmou que são compras online e no exterior, o que levou às colunas `channel`, `merchant_country` e `alert_type_mismatch`.

In [0]:
%sql
CREATE OR REPLACE TABLE financial_transactions.silver.transactions AS
SELECT
  CAST(id AS BIGINT)                                AS id,
  TO_TIMESTAMP(`date`, 'yyyy-MM-dd HH:mm:ss')       AS transaction_date,
  CAST(client_id AS INT)                            AS client_id,
  CAST(card_id AS INT)                              AS card_id,
  CAST(REPLACE(amount, '$', '') AS DECIMAL(12,2))   AS amount,
  use_chip                                          AS transaction_type,
  CASE
    WHEN merchant_city = 'ONLINE'    THEN 'Online'
    WHEN LENGTH(merchant_state) = 2  THEN 'Presencial EUA'
    ELSE 'Presencial exterior'
  END                                               AS channel,
  CAST(merchant_id AS INT)                          AS merchant_id,
  merchant_city,
  CASE WHEN LENGTH(merchant_state) = 2 THEN merchant_state END AS merchant_state,
  CASE
    WHEN merchant_city = 'ONLINE'    THEN NULL
    WHEN LENGTH(merchant_state) = 2  THEN 'United States'
    ELSE merchant_state
  END                                               AS merchant_country,
  LPAD(CAST(CAST(zip AS DOUBLE) AS INT), 5, '0')    AS zip,
  CAST(mcc AS INT)                                  AS mcc,
  COALESCE(errors, 'Nenhum')                        AS errors,
  (use_chip <> 'Online Transaction' AND merchant_city = 'ONLINE') AS alert_type_mismatch,
  _data_ingestao,
  CURRENT_TIMESTAMP()                               AS _data_processamento
FROM financial_transactions.bronze.transactions

num_affected_rows,num_inserted_rows


**Ideal:** Quantidade de linhas iguais e 0 datas, valores e erros vazios.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM financial_transactions.bronze.transactions) AS linhas_bronze,
  COUNT(*)                            AS linhas_silver,
  COUNT_IF(transaction_date IS NULL)  AS data_vazia,
  COUNT_IF(amount IS NULL)            AS valor_vazio,
  COUNT_IF(errors IS NULL)            AS erros_vazios,
  COUNT_IF(alert_type_mismatch)       AS alertas_tipo
FROM financial_transactions.silver.transactions

linhas_bronze,linhas_silver,data_vazia,valor_vazio,erros_vazios,alertas_tipo
13305915,13305915,0,0,0,5788


### 4. Formatação e tratamento: mcc_codes
Converte o código para número, para ligar com as transações, e traduz o nome da coluna de descrição para o inglês, no padrão das outras tabelas.


In [0]:
%sql
CREATE OR REPLACE TABLE financial_transactions.silver.mcc_codes AS
SELECT
  CAST(mcc AS INT)     AS mcc,
  descricao            AS description,
  _data_ingestao,
  CURRENT_TIMESTAMP()  AS _data_processamento
FROM financial_transactions.bronze.mcc_codes

num_affected_rows,num_inserted_rows


**Ideal:** Quantidade de linhas iguais e 0 códigos ou descrições vazias.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM financial_transactions.bronze.mcc_codes) AS linhas_bronze,
  COUNT(*)                        AS linhas_silver,
  COUNT_IF(mcc IS NULL)           AS codigo_vazio,
  COUNT_IF(description IS NULL)   AS descricao_vazia
FROM financial_transactions.silver.mcc_codes

linhas_bronze,linhas_silver,codigo_vazio,descricao_vazia
109,109,0,0


### 5. Formatação e tratamento: fraud_labels
Converte o id da transação para número e o rótulo "Yes/No" para verdadeiro/falso.

> **Ajuste após a validação:** na primeira versão, as duas colunas ficaram como texto; a verificação de formato encontrou o problema e a tabela foi recriada com as conversões corretas.

In [0]:
%sql
CREATE OR REPLACE TABLE financial_transactions.silver.fraud_labels AS
SELECT
  CAST(transaction_id AS BIGINT)  AS transaction_id,
  is_fraud = 'Yes'                AS is_fraud,
  _data_ingestao,
  CURRENT_TIMESTAMP()             AS _data_processamento
FROM financial_transactions.bronze.fraud_labels

num_affected_rows,num_inserted_rows


**Ideal:** Quantidade de linhas iguais, 0 ids e rótulos vazios.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM financial_transactions.bronze.fraud_labels) AS linhas_bronze,
  COUNT(*)                          AS linhas_silver,
  COUNT_IF(transaction_id IS NULL)  AS id_vazio,
  COUNT_IF(is_fraud IS NULL)        AS rotulo_vazio,
  COUNT_IF(is_fraud)                AS fraudes
FROM financial_transactions.silver.fraud_labels

linhas_bronze,linhas_silver,id_vazio,rotulo_vazio,fraudes
8914963,8914963,0,0,13332


### 6. Quarentena: transações de valor zero
Separa as transações com valor zero em uma tabela própria no schema quality, com o motivo e a data, e as retira da Silver. Elas não entram nas análises de risco, mas continuam guardadas para consulta.

In [0]:
%sql
CREATE OR REPLACE TABLE financial_transactions.quality.quarentena_transactions AS
SELECT
  *,
  'valor igual a zero' AS motivo_quarentena,
  CURRENT_TIMESTAMP()  AS _data_quarentena
FROM financial_transactions.silver.transactions
WHERE amount = 0

num_affected_rows,num_inserted_rows


Retira da Silver as transações enviadas para a quarentena.

In [0]:
%sql
DELETE FROM financial_transactions.silver.transactions
WHERE amount = 0

num_affected_rows
10639


**Ideal:** a soma das linhas da Silver e da quarentena é igual ao total da Bronze, e nenhuma transação de valor zero ficou na Silver.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM financial_transactions.bronze.transactions)             AS linhas_bronze,
  (SELECT COUNT(*) FROM financial_transactions.silver.transactions)             AS linhas_silver,
  (SELECT COUNT(*) FROM financial_transactions.quality.quarentena_transactions) AS linhas_quarentena,
  (SELECT COUNT_IF(amount = 0) FROM financial_transactions.silver.transactions) AS valor_zero_na_silver

linhas_bronze,linhas_silver,linhas_quarentena,valor_zero_na_silver
13305915,13295276,10639,0


### 7. Quarentena: rótulos de fraude
Separa os rótulos de fraude das transações em quarentena em uma tabela própria no schema quality e os retira da Silver, para que todo rótulo da Silver tenha uma transação correspondente.

In [0]:
%sql
CREATE OR REPLACE TABLE financial_transactions.quality.quarentena_fraud_labels AS
SELECT
  f.*,
  'transação em quarentena' AS motivo_quarentena,
  CURRENT_TIMESTAMP()       AS _data_quarentena
FROM financial_transactions.silver.fraud_labels f
WHERE f.transaction_id IN (SELECT id FROM financial_transactions.quality.quarentena_transactions)

num_affected_rows,num_inserted_rows


Retira da Silver os rótulos enviados para a quarentena.

In [0]:
%sql
DELETE FROM financial_transactions.silver.fraud_labels
WHERE transaction_id IN (SELECT id FROM financial_transactions.quality.quarentena_transactions)

num_affected_rows
7193


**Ideal:** a soma dos rótulos da Silver e da quarentena é igual ao total da Bronze, e nenhum rótulo da Silver ficou sem transação.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM financial_transactions.bronze.fraud_labels)             AS linhas_bronze,
  (SELECT COUNT(*) FROM financial_transactions.silver.fraud_labels)             AS linhas_silver,
  (SELECT COUNT(*) FROM financial_transactions.quality.quarentena_fraud_labels) AS linhas_quarentena,
  (SELECT COUNT(*) FROM financial_transactions.silver.fraud_labels f
     LEFT JOIN financial_transactions.silver.transactions t ON f.transaction_id = t.id
   WHERE t.id IS NULL)                                                           AS sem_transacao

linhas_bronze,linhas_silver,linhas_quarentena,sem_transacao
8914963,8907770,7193,0
